# Neural Machine Translation Demo with Marian
## Sequence to Sequence Architecture

**By Ye Kyaw Thu, Lab Leader, Language Understanding Lab., Myanmar**  
**Date:** 24 May 2026  
*For AI (Fundamental) Class students*  

## Colab Adaptation Note

**Student: Yee Mon Thant**

This copy has been adapted to run on **Google Colab**. Only two things were changed from the original notebook by Ye Kyaw Thu: (1) Marian is loaded from a **pre-built binary already saved on Google Drive** instead of being installed/compiled here, and (2) the **training/dev/test data files are uploaded** at the point they're needed, since Colab starts with an empty filesystem. All training hyperparameters, commands, and explanations below are otherwise unchanged from the original.

**Note on structure:** Everything up to and including the "Evaluation on Seq2Seq Model" section follows Sayar's original notebook exactly, with only the Colab infrastructure adaptations noted above. The "Task 4: Improving the Seq2Seq Baseline" section that follows is my own addition, written to fulfill the assignment's requirement to improve on the baseline result.



In [ ]:
# Set up the working directory (equivalent to the teacher's marian-demo root folder)
import os
os.makedirs('/content/marian-demo/g2p-par', exist_ok=True)
%cd /content/marian-demo

/content/marian-demo


In [ ]:
# Load the CUDA-enabled Marian binaries from Google Drive (see marian_build_colab-GPU.ipynb)
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/marian-bin-gpu', exist_ok=True)

for binary in ['marian', 'marian-decoder', 'marian-vocab']:
    shutil.copy(f'/content/drive/MyDrive/marian-build-gpu/{binary}', '/content/marian-bin-gpu/')
    os.chmod(f'/content/marian-bin-gpu/{binary}', 0o755)

os.environ['PATH'] = '/content/marian-bin-gpu:' + os.environ['PATH']
print('CUDA-enabled Marian ready ✅ (make sure Runtime > Change runtime type = GPU)')

Mounted at /content/drive
CUDA-enabled Marian ready ✅ (make sure Runtime > Change runtime type = GPU)


## Marian Installation

အရင်ဆုံး ကိုယ့်စက်ထဲမှာ marian ကို installation လုပ်ပါ။  
Marian homepage: [https://github.com/marian-nmt/marian](https://github.com/marian-nmt/marian)  

Installation လုပ်ထားပြီးသွားရင် marian command တွေကို ခေါ်သုံးလို့ ရပါပြီ။  


In [ ]:
!marian --help

Marian: Fast Neural Machine Translation in C++
Usage: marian [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=2048               Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), critical, off
  --log-time-zone TEXT  

**အထက်မှာ မြင်ရတဲ့အတိုင်းပါပဲ၊ neural network  နဲ့ဆိုင်တဲ့ hyperparameter အပါအဝင် တခြား command line option တွေ အများကြီးရှိပါတယ်။**  

Vocab ဆောက်ဖို့အတွက်ကတော့ marian-vocab ဆိုတဲ့ command ကို သုံးပါမယ်။  

In [ ]:
!marian-vocab --help

Create a vocabulary from text corpora given on STDIN
Usage: marian-vocab [OPTIONS]

Allowed options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  -m,--max-size UINT=0                  Generate only UINT most common vocabulary items

Examples:
  ./marian-vocab < text.src > vocab.yml
  cat text.src text.trg | ./marian-vocab > vocab.yml


## Data Preparation

SMT assignment အတွက် သုံးထားတဲ့ grapheme to phoneme (g2p) ဒေတာကိုပဲ သုံးပြီး NMT architecture နှစ်မျိုးကိုသုံးပြီး machine translation လုပ်ပြပါမယ်။ စာကြောင်း အနေနဲ့က syllable ဖြတ်ထားတဲ့ word တွေမို့လို့ length သိပ်မရှည်ပါဘူး။ အဲဒါကြောင့် ဆရာ အခုသုံးပြမယ့် GPU machine မှာတော့ training time က မြန်ပါလိမ်မယ်။  



In [ ]:
%pwd

'/content/marian-demo'

In [ ]:
!ls --color=auto

g2p-par


In [ ]:
%cd g2p-par

/content/marian-demo/g2p-par


### Colab Data Upload



In [ ]:
from google.colab import files

required = {'train.my', 'train.ph', 'dev.my', 'dev.ph', 'test.my', 'test.ph'}
existing = set(os.listdir('.'))

if required.issubset(existing):
    print('Data files already present, skipping upload.')
else:
    print('Please select the 6 g2p-par data files: train.my, train.ph, dev.my, dev.ph, test.my, test.ph')
    uploaded = files.upload()
    missing = required - set(os.listdir('.'))
    if missing:
        print(f'⚠️ Still missing: {missing}')
    else:
        print('✅ All 6 data files present.')

Please select the 6 g2p-par data files: train.my, train.ph, dev.my, dev.ph, test.my, test.ph


Saving dev.my to dev.my
Saving dev.ph to dev.ph
Saving test.my to test.my
Saving test.ph to test.ph
Saving train.my to train.my
Saving train.ph to train.ph
✅ All 6 data files present.


In [ ]:
!ls *

dev.my	dev.ph	test.my  test.ph  train.my  train.ph


### Vocab Building  

Vocab ကို ဆောက်ဖို့အတွက် training ဖိုင်နဲ့ validation or development ဖိုင်ကို ပေါင်းပါမယ်။  
အဲဒီ အလုပ်ကို source language အတွက်ကော၊ target language အတွက်ကော လုပ်ရမယ်။  
အရင်ဆုံး preprocessing ဆိုတာဆောက်လိုက်ပြီး အဲဒီအထဲမှာ ပေါင်းထားတဲ့ ဖိုင်နှစ်ဖိုင်ကို သိမ်းကြရအောင်။  

In [ ]:
!mkdir preprocessing

In [ ]:
!cat train.my dev.my > ./preprocessing/train-dev.my

In [ ]:
!cat train.ph dev.ph > ./preprocessing/train-dev.ph

In [ ]:
!wc ./preprocessing/*.{my,ph}

 22000  63023 653405 ./preprocessing/train-dev.my
 22000  63034 286205 ./preprocessing/train-dev.ph
 44000 126057 939610 total


မြန်မာစာ အတွက် vocab ဖိုင်ကို ဆောက်မယ်။  

In [ ]:
!mkdir vocab

In [ ]:
!marian-vocab < ./preprocessing/train-dev.my > ./vocab/vocab.my.yml

[2026-08-06 09:34:22] Creating vocabulary...
[2026-08-06 09:34:22] [data] Creating vocabulary stdout from stdin
[2026-08-06 09:34:22] Finished


Phoneme ပိုင်းအတွက်လည်း vocab ဖိုင်ကို ဆောက်ပါမယ်။  

In [ ]:
!marian-vocab < ./preprocessing/train-dev.ph > ./vocab/vocab.ph.yml

[2026-08-06 09:34:23] Creating vocabulary...
[2026-08-06 09:34:23] [data] Creating vocabulary stdout from stdin
[2026-08-06 09:34:23] Finished


Vocab ဖိုင်ရဲ့ format ကိုလေ့လာကြည့်ကြရအောင်။ တကယ်ကတော့ ကိုယ့် training/validation ဖိုင်ထဲမှာ ပါဝင်တဲ့ စာလုံးတစ်လုံးချင်းစီကို (i.e. unit words) နံပါတ်ထိုးပြီး သိမ်းထားတဲ့ ပုံစံပါပဲ။  

In [ ]:
!head -n 30 ./vocab/vocab.my.yml

</s>: 0
<unk>: 1
အ: 2
မ: 3
သ: 4
က: 5
တ: 6
လက်: 7
ပ: 8
စ: 9
စာ: 10
ရေ: 11
ရ: 12
ရာ: 13
ကြီး: 14
တစ်: 15
စား: 16
သား: 17
နာ: 18
ခ: 19
ခံ: 20
မီး: 21
လူ: 22
မျက်: 23
ကျ: 24
သာ: 25
သံ: 26
ပေါက်: 27
လုံး: 28
ကာ: 29


In [ ]:
!head -n 30 ./vocab/vocab.ph.yml

</s>: 0
<unk>: 1
a-: 2
ma-: 3
da-: 4
le': 5
jei: 6
ga-: 7
ta-: 8
tha-: 9
ja: 10
ba-: 11
dha-: 12
za-: 13
pa-: 14
na: 15
"mi:": 16
mje': 17
ka-: 18
than: 19
"gyi:": 20
za: 21
lu: 22
ja.: 23
ma.: 24
le: 25
sa: 26
na-: 27
kha-: 28
"loun:": 29


## Preparing a Shell Script for Phoneme to Myanmar Translation

Phoneme to Myanmar conversion ကို neural machine translation နဲ့ လုပ်ကြည့်ပြီး performance ကို တိုင်းတာကြည့်ကြရအောင်။  

အဲဒီလို လုပ်ဖို့အတွက် ဆရာကတော့ shell script ကို ပြင်ပြီး run တဲ့ ပုံစံနဲ့ပဲ သွားပါတယ်။  
Source language ကို phoneme ထားပြီး၊ target language ကို grapheme ထားမယ်။  

အရင်ဆုံး sequence to sequence အာခီတက်ချာနဲ့ပဲ သွားမယ်။  

In [ ]:
%cd /content/marian-demo

/content/marian-demo


### Colab Adaptation: recreate `seq2seq.phmy.sh`

This script didn't exist in the uploaded files, so it's recreated here using the same hyperparameters shown elsewhere in the teacher's notebook. Only two things changed: the data path (points to the uploaded `g2p-par` folder), and it uses the GPU-enabled Marian binary with `--devices 0` for training.

In [ ]:
%%writefile ./seq2seq.phmy.sh
export PATH=/content/marian-bin-gpu:$PATH

model_folder="model.seq2seq.phmy"
mkdir -p ${model_folder}
data_path="/content/marian-demo/g2p-par"
src="ph"; tgt="my"

marian \
  --type s2s \
  --train-sets ${data_path}/train.${src} ${data_path}/train.${tgt} \
  --max-length 200 \
  --valid-sets ${data_path}/dev.${src} ${data_path}/dev.${tgt} \
  --vocabs ${data_path}/vocab/vocab.${src}.yml ${data_path}/vocab/vocab.${tgt}.yml \
  --model ${model_folder}/model.npz \
  --workspace 500 \
  --enc-depth 2 --enc-type alternating --enc-cell lstm --enc-cell-depth 2 \
  --dec-depth 2 --dec-cell lstm --dec-cell-base-depth 2 --dec-cell-high-depth 2 \
  --tied-embeddings --layer-normalization --skip \
  --mini-batch-fit --valid-mini-batch 32 \
  --valid-metrics cross-entropy perplexity bleu \
  --valid-freq 5000 --save-freq 5000 --disp-freq 500 \
  --dropout-rnn 0.3 --dropout-src 0.3 --exponential-smoothing \
  --early-stopping 10 \
  --devices 0 --seed 1111 \
  --dump-config > ${model_folder}/config.yml

time marian -c ${model_folder}/config.yml 2>&1 | tee ${model_folder}/train.log

Writing ./seq2seq.phmy.sh


In [ ]:
!chmod +x ./seq2seq.phmy.sh

In [ ]:
from IPython.display import Markdown

# Read the file content
with open('./seq2seq.phmy.sh', 'r') as f:
    content = f.read()

# Display it as a highlighted Bash block
display(Markdown(f"```bash\n{content}\n```"))

```bash
export PATH=/content/marian-bin-gpu:$PATH

model_folder="model.seq2seq.phmy"
mkdir -p ${model_folder}
data_path="/content/marian-demo/g2p-par"
src="ph"; tgt="my"

marian \
  --type s2s \
  --train-sets ${data_path}/train.${src} ${data_path}/train.${tgt} \
  --max-length 200 \
  --valid-sets ${data_path}/dev.${src} ${data_path}/dev.${tgt} \
  --vocabs ${data_path}/vocab/vocab.${src}.yml ${data_path}/vocab/vocab.${tgt}.yml \
  --model ${model_folder}/model.npz \
  --workspace 500 \
  --enc-depth 2 --enc-type alternating --enc-cell lstm --enc-cell-depth 2 \
  --dec-depth 2 --dec-cell lstm --dec-cell-base-depth 2 --dec-cell-high-depth 2 \
  --tied-embeddings --layer-normalization --skip \
  --mini-batch-fit --valid-mini-batch 32 \
  --valid-metrics cross-entropy perplexity bleu \
  --valid-freq 5000 --save-freq 5000 --disp-freq 500 \
  --dropout-rnn 0.3 --dropout-src 0.3 --exponential-smoothing \
  --early-stopping 10 \
  --devices 0 --seed 1111 \
  --dump-config > ${model_folder}/config.yml

time marian -c ${model_folder}/config.yml 2>&1 | tee ${model_folder}/train.log

```

အရေးကြီးတဲ့ parameter တွေကိုပဲ ဆရာ သုံးပြထားပါတယ်။ ကျောင်းသားတွေက ကိုယ်စက်ထဲမှာ run မယ်ဆိုရင်တော့ အဓိက ပြင်ရမှာက path တွေပါပဲ။

```bash
model_folder="model.seq2seq.phmy";
mkdir ${model_folder};
data_path="/home/ye/exp/nmt/marian-demo/g2p-par";
src="ph"; tgt="my";
```

အာခီတက်ချာကိုတော့ type ဆိုတဲ့ option နဲ့ ပြောင်းလို့ ရပါတယ်။  

```bash
--type s2s
```

ပြီးတော့ GPU က တစ်လုံးထက်ပိုရင် 0 1 2 3 ဆိုပြီး ဖြည့်တာမျိုး လုပ်ပေးရလိမ့်မယ်။ 0 to 3 ပေးထားရင် ကိုယ့်စက်ထဲမှာ ရှိတဲ့ GPU ကဒ်လေးကဒ်စလုံးကို အသုံးပြုမယ်လို့ ဆိုလိုတာပါ။  

အများသောအားဖြင့်က Neural network မော်ဒယ်တွေကို training လုပ်တဲ့အခါမှာ play ရတဲ့ batch size, memory ပိုင်း, hidden layer အရေအတွက်, epoch, dropout စတာတွေပဲ ဖြစ်ပါတယ်။ Marian framework မှာဆိုရင်တော့ အများသောအားဖြင့် error တက်တာက -workspace, --enc-depth, --decc-depth တွေကလည်း အရေးကြီးပါတယ်။ Learning လုပ်နေတဲ့အချိန်မှာ ဘယ်လို evaluation metric နဲ့ ထားမလဲ ဆိုတာကိုတော့ --valid-metrics ဆိုတဲ့ command line option နဲ့ ချိန်လို့ ရပါတယ်။ ဆရာကတော့ အထက်မှာ မြင်ရတဲ့အတိုင်း `--valid-metrics cross-entropy perplexity bleu` သုံးခု ထားပါတယ်။ ပြီးတော့ ကိုယ့် HDD ထဲမှာ မော်ဒယ်တွေ အများကြီးသိမ်းရင် hardisk space ယူတာမို့လို့ ဆရာကတော့ `-save-freq 5000` ဆိုတဲ့ setting ကို လုပ်ထားပါတယ်။ အခြေခံအားဖြင့် ဆရာပြင်ပေးထားတဲ့ setting ဖိုင်နဲ့ အဆင်ပြေပါလိမ့်မယ်။ သို့သော် ကိုယ့်စက်ရဲ့ GPU, memory ပြီးတော့ တခြား လိုချင်တဲ့ training/tuning parameter တွေကတော့ ကိုယ်တိုင် run ကြည့်လိုက် ရလဒ်ကို ကြည့်လိုက်လုပ်ပြီး ညှိယူရပါလိမ့်မယ်။ အထူးသဖြင့် ကိုယ့် ဒေတာပမာဏ၊ ဒိုမိန်းနဲ့ အကောင်းဆုံး NMT performance ကို ရဖို့အတွက်က တစ်ကြိမ်ထက်မက training လုပ်ကြရပါလိမ့်မယ်။    

Shell script မှာက နှစ်ပိုင်း ပါဝင်ပါတယ်။ တစ်ပိုင်း (သို့) ပထမ command က configuration ဖိုင် တည်ဆောက်တာပါ။ နောက်တစ်ပိုင်း က ဆောက်ထားတဲ့ configuration ဖိုင်ကို သုံးပြီးတော့ NMT လုပ်တဲ့ အပိုင်းပါ။  



## Seq2Seq Training for Phoneme to Grapheme


In [ ]:
!./seq2seq.phmy.sh

[2026-08-06 09:34:23] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-08-06 09:34:23] [marian] Running on a0b09224bf18 as process 1485 with command line:
[2026-08-06 09:34:23] [marian] marian -c model.seq2seq.phmy/config.yml
[2026-08-06 09:34:23] [config] after: 0e
[2026-08-06 09:34:23] [config] after-batches: 0
[2026-08-06 09:34:23] [config] after-epochs: 0
[2026-08-06 09:34:23] [config] all-caps-every: 0
[2026-08-06 09:34:23] [config] allow-unk: false
[2026-08-06 09:34:23] [config] authors: false
[2026-08-06 09:34:23] [config] beam-size: 12
[2026-08-06 09:34:23] [config] bert-class-symbol: "[CLS]"
[2026-08-06 09:34:23] [config] bert-mask-symbol: "[MASK]"
[2026-08-06 09:34:23] [config] bert-masking-fraction: 0.15
[2026-08-06 09:34:23] [config] bert-sep-symbol: "[SEP]"
[2026-08-06 09:34:23] [config] bert-train-type-embeddings: true
[2026-08-06 09:34:23] [config] bert-type-vocab-size: 2
[2026-08-06 09:34:23] [config] build-info: ""
[2026-08-06 09:34:23] [config] check-g

### Save Trained Model to Google Drive


In [ ]:
import shutil, os
drive_save_path = '/content/drive/MyDrive/marian-results/seq2seq.phmy'
os.makedirs(os.path.dirname(drive_save_path), exist_ok=True)
shutil.copytree('./model.seq2seq.phmy', drive_save_path, dirs_exist_ok=True)
print(f'✅ Saved to {drive_save_path}')

✅ Saved to /content/drive/MyDrive/marian-results/seq2seq.phmy


## Configuration File for Ph-My

In [ ]:
!cat ./model.seq2seq.phmy/config.yml

# Marian configuration file generated at 2026-08-06 09:34:23 +0000 with version v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
# General options
authors: false
cite: false
build-info: ""
workspace: 500
log: ""
log-level: info
log-time-zone: ""
quiet: false
quiet-translation: false
seed: 1111
check-nan: false
interpolate-env-vars: false
relative-paths: false
sigterm: save-and-exit
# Model options
model: model.seq2seq.phmy/model.npz
pretrained-model: ""
ignore-model-config: false
type: s2s
dim-vocabs:
  - 0
  - 0
dim-emb: 512
factors-dim-emb: 0
factors-combine: sum
lemma-dependency: ""
lemma-dim-emb: 0
dim-rnn: 1024
enc-type: alternating
enc-cell: lstm
enc-cell-depth: 2
enc-depth: 2
dec-cell: lstm
dec-cell-base-depth: 2
dec-cell-high-depth: 2
dec-depth: 2
skip: true
layer-normalization: true
right-left: false
input-types:
  []
best-deep: false
tied-embeddings: true
tied-embeddings-src: false
tied-embeddings-all: false
output-omit-bias: false
transformer-heads: 8
transformer-no-projection: fa

## Checking Output Models

In [ ]:
!ls ./model.seq2seq.phmy/ --color=auto

config.yml           model.iter40000.npz  model.npz.decoder.yml
model.iter10000.npz  model.iter45000.npz  model.npz.optimizer.npz
model.iter15000.npz  model.iter50000.npz  model.npz.progress.yml
model.iter20000.npz  model.iter5000.npz   model.npz.yml
model.iter25000.npz  model.iter55000.npz  train.log
model.iter30000.npz  model.iter60000.npz
model.iter35000.npz  model.npz


အထက်မှာ မြင်ရတဲ့ npz ဖိုင်တွေက မော်ဒယ် ဖိုင်တွေပါပဲ။ ဆရာက iteration 5000 စီမှာ တစ်ခေါက်စီ သိမ်းထားတဲ့ setting နဲ့ run ခဲ့တာပါ။ အဲဒီအထဲကမှ training process တစ်ခုလုံးအတွက် အကောင်းဆုံး မော်ဒယ်ဖိုင်က model.npz ဖိုင်ပါပဲ။  

training log ဖိုင်တွေ validation log ဖိုင်တွေကိုလည်း လေ့လာကြည့်ပါ။  

## Testing with Seq2Seq Model for Phoneme to Grapheme Translation

အရင်ဆုံး GPU က အားမအားလည်း စစ်ကြည့်ပါ။  

In [ ]:
!nvidia-smi

Thu Aug  6 10:55:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!time marian-decoder -m ./model.seq2seq.phmy/model.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0 < ./g2p-par/test.ph > ./seq2seq.phmy.hyp.txt

[2026-08-06 10:55:00] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-08-06 10:55:00] [marian] Running on a0b09224bf18 as process 22494 with command line:
[2026-08-06 10:55:00] [marian] marian-decoder -m ./model.seq2seq.phmy/model.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0
[2026-08-06 10:55:01] [config] alignment: ""
[2026-08-06 10:55:01] [config] allow-special: false
[2026-08-06 10:55:01] [config] allow-unk: false
[2026-08-06 10:55:01] [config] authors: false
[2026-08-06 10:55:01] [config] beam-size: 12
[2026-08-06 10:55:01] [config] bert-class-symbol: "[CLS]"
[2026-08-06 10:55:01] [config] bert-mask-symbol: "[MASK]"
[2026-08-06 10:55:01] [config] bert-masking-fraction: 0.15
[2026-08-06 10:55:01] [config] bert-sep-symbol: "[SEP]"
[2026-08-06 10:55:01] [config] bert-train-type-embeddings: true
[2026-08-06 10:55:01] [config] bert-type-vocab-size: 2
[2026-08-06 10:55:01] [config] best-deep: false
[2026-08-06 10:55:01] [config] build-info

## Checking Hypothesis File  

Test data နဲ့ translation လုပ်ပြီး ရလာတဲ့ hypothesis ဖိုင်ကို လေ့လာကြည့်ကြရအောင်။  

In [ ]:
!head -n 30 ./seq2seq.phmy.hyp.txt

တက် တက် ပြောင်
ကပ် ပိ
ရှုံ့ မဲ့
ညှဉ်း ပန်း
မွမ်း မံ
ငယ် မည်
ရှာ ရှာ ဖွေ ဗွေ
ကံ ပြင်း
ဥ မ ကွဲ သိုက် မ ပျက်
အ စစ်
ရေ ကျ
မ ဆုတ် မ ဆိုင်း
ဟ ကွက်
မိန်း မူး
ကယ် မ
နင်း နယ်
က ကြိုး ဒ စာ
ဆွမ်း ကြီး လောင်း
ဝါ ကျင့် ကျင့်
ကုတ် ဟဲ နာ
နီ ကြင် ကြင်
အ လို တူ
ကိန်း ရင်း
မီး တိုင်
ဝေ့ လည် ကြောင် ဘတ်
စ ကား ပြော ကြေး နန်း
အ ချစ် ဦး
အ သည်း ကောင်း
ည ကြီး
ချို မိုင် မြ


**Test input ဖိုင်နဲ့ တွဲကြည့်ကြည့်ရအောင်။**  

In [ ]:
!paste ./g2p-par/test.ph ./seq2seq.phmy.hyp.txt | head -n 30

te' te' pjaun	တက် တက် ပြောင်
ka' pi.	ကပ် ပိ
shoun. me.	ရှုံ့ မဲ့
njhin: ban:	ညှဉ်း ပန်း
mun: man	မွမ်း မံ
nge mji	ငယ် မည်
sha sha hpwei bwei	ရှာ ရှာ ဖွေ ဗွေ
ba- bjin:	ကံ ပြင်း
u. ma- kwe: thai' ma- pje'	ဥ မ ကွဲ သိုက် မ ပျက်
a- si'	အ စစ်
jei kya.	ရေ ကျ
ma- hsou' ma- hsain:	မ ဆုတ် မ ဆိုင်း
ha. gwe'	ဟ ကွက်
mein: mu:	မိန်း မူး
ke ma.	ကယ် မ
nin: ne	နင်း နယ်
ka. gyou: da- za	က ကြိုး ဒ စာ
hsun: gyi: laun:	ဆွမ်း ကြီး လောင်း
wa kyin. gyin.	ဝါ ကျင့် ကျင့်
kou' he: na	ကုတ် ဟဲ နာ
ni kyin gyin	နီ ကြင် ကြင်
a- lou tu	အ လို တူ
kein: jin:	ကိန်း ရင်း
mi: dain	မီး တိုင်
wei. le gyaun ba'	ဝေ့ လည် ကြောင် ဘတ်
za- ga: bjo: kyei: nan:	စ ကား ပြော ကြေး နန်း
a- chi' u:	အ ချစ် ဦး
a- the: kaun:	အ သည်း ကောင်း
nja. gyi:	ည ကြီး
chou mja mja.	ချို မိုင် မြ


## Evaluation on Seq2Seq Model

SMT demo တုန်းက သုံးခဲ့တဲ့ BLEU socre တွက်တဲ့ perl script ကိုပဲ သုံးပြီး evaluation လုပ်ကြည့်ပါမယ်။  

### Colab Adaptation: get `multi-bleu.perl`



In [ ]:
!wget -nc -q https://raw.githubusercontent.com/moses-smt/mosesdecoder/master/scripts/generic/multi-bleu.perl -O ./multi-bleu.perl

In [ ]:
!perl ./multi-bleu.perl ./g2p-par/test.my < ./seq2seq.phmy.hyp.txt

BLEU = 77.90, 87.8/79.4/74.7/70.9 (BP=1.000, ratio=1.000, hyp_len=8044, ref_len=8047)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.


## Task 4: Improving the Seq2Seq Baseline

I first tried fine-tuning the baseline model with label smoothing and a learning-rate warmup (script below), hoping to push BLEU higher.

In [ ]:
%%writefile ./seq2seq.phmy.v2.sh
export PATH=/content/marian-bin-gpu:$PATH

model_folder="model.seq2seq.phmy.v2"
rm -rf ${model_folder}
mkdir -p ${model_folder}
data_path="/content/marian-demo/g2p-par"
src="ph"; tgt="my"

cp ./model.seq2seq.phmy/model.npz ${model_folder}/model.npz

marian \
  --type s2s \
  --train-sets ${data_path}/train.${src} ${data_path}/train.${tgt} \
  --max-length 200 \
  --valid-sets ${data_path}/dev.${src} ${data_path}/dev.${tgt} \
  --vocabs ${data_path}/vocab/vocab.${src}.yml ${data_path}/vocab/vocab.${tgt}.yml \
  --model ${model_folder}/model.npz \
  --workspace 500 \
  --enc-depth 2 --enc-type alternating --enc-cell lstm --enc-cell-depth 2 \
  --dec-depth 2 --dec-cell lstm --dec-cell-base-depth 2 --dec-cell-high-depth 2 \
  --tied-embeddings --layer-normalization --skip \
  --mini-batch-fit --valid-mini-batch 32 \
  --valid-metrics cross-entropy perplexity bleu \
  --valid-freq 250 --save-freq 250 --disp-freq 50 \
  --dropout-rnn 0.3 --dropout-src 0.3 --exponential-smoothing \
  --label-smoothing 0.1 \
  --learn-rate 0.0001 \
  --after-batches 500 \
  --devices 0 --seed 1111 \
  --dump-config > ${model_folder}/config.yml

time marian -c ${model_folder}/config.yml 2>&1 | tee ${model_folder}/train.log

Overwriting ./seq2seq.phmy.v2.sh


In [ ]:
!chmod +x ./seq2seq.phmy.v2.sh
!./seq2seq.phmy.v2.sh

### Improving the baseline without retraining: using the best checkpoint

Since fine-tuning made things worse (see above), I tried a different, safer approach: Marian already saves a checkpoint every 5,000 updates during training (`model.iter5000.npz`, `model.iter10000.npz`, ... up to `model.iter60000.npz`), in addition to the final `model.npz`. Looking back at the baseline training log, validation BLEU peaked around iteration 20,000 and slightly declined afterward — a sign of mild overfitting by the time training finished.

So instead of using the final `model.npz` (the one used for the baseline evaluation above), I decode with `model.iter20000.npz` — the checkpoint saved at that peak. This requires no additional training at all; it's simply choosing a better-performing snapshot of the same model.

In [ ]:
!time marian-decoder -m ./model.seq2seq.phmy/model.iter20000.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0 < ./g2p-par/test.ph > ./seq2seq.phmy.best-ckpt.hyp.txt
!perl ./multi-bleu.perl ./g2p-par/test.my < ./seq2seq.phmy.best-ckpt.hyp.txt

[2026-08-06 11:32:17] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-08-06 11:32:17] [marian] Running on a0b09224bf18 as process 32510 with command line:
[2026-08-06 11:32:17] [marian] marian-decoder -m ./model.seq2seq.phmy/model.iter20000.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0
[2026-08-06 11:32:17] [config] alignment: ""
[2026-08-06 11:32:17] [config] allow-special: false
[2026-08-06 11:32:17] [config] allow-unk: false
[2026-08-06 11:32:17] [config] authors: false
[2026-08-06 11:32:17] [config] beam-size: 12
[2026-08-06 11:32:17] [config] bert-class-symbol: "[CLS]"
[2026-08-06 11:32:17] [config] bert-mask-symbol: "[MASK]"
[2026-08-06 11:32:17] [config] bert-masking-fraction: 0.15
[2026-08-06 11:32:17] [config] bert-sep-symbol: "[SEP]"
[2026-08-06 11:32:17] [config] bert-train-type-embeddings: true
[2026-08-06 11:32:17] [config] bert-type-vocab-size: 2
[2026-08-06 11:32:17] [config] best-deep: false
[2026-08-06 11:32:17] [config] 

In [ ]:
import shutil, os
drive_save_path = '/content/drive/MyDrive/marian-results/seq2seq.phmy.best'
os.makedirs(drive_save_path, exist_ok=True)
shutil.copy('./model.seq2seq.phmy/model.iter20000.npz', f'{drive_save_path}/model.iter20000.npz')
print(f'✅ Saved best checkpoint to {drive_save_path}')

✅ Saved best checkpoint to /content/drive/MyDrive/marian-results/seq2seq.phmy.best


### Evaluate and compare to the baseline BLEU score above

### Comparison: Baseline vs. Improved

| Model | BLEU |
|---|---|
| Baseline (final `model.npz`) | 77.90 |
| Improved (best checkpoint, `model.iter20000.npz`) | 78.47 |

